**The purpose of this notebook:** 
apply the clean function and attend the specific needs of the dataset to The final cleaned dataset to be used for further analysis and modeling.


In [1]:
import pandas as pd
import yaml
import openpyxl

---

In [3]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config").is_dir())
with open(ROOT / "config" / "config.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
cfg = {k: str(ROOT / v) for k, v in cfg.items()}  # relative paths resolve against the project root

In [4]:
P_B1_24 = pd.read_excel(cfg["P_B1_24_path"], engine="openpyxl")

---

In [5]:
import sys
sys.path.append("../..")
from src.functions import clean_plant

In [6]:
P_B1_24_clean = clean_plant(P_B1_24)
P_B1_24_clean

Rows with all NaN values: 1286
Rows with 'Dosif. m3' in first column: 1286


,\n \n Fecha,\n \n Fórmula,\n \n Volumen,Cemento\n 4\n CEM III/A 42,Arido\n 1\n ARENA 04,Arido\n 2\n ARIDO 8/14,Arido\n 3\n ARIDO 12/20,Arido\n 4\n ARENA CORREC,Agua\n 1\n AGUA,Aditivo\n 1\n AURAMIX 269,Aditivo\n 2\n DYNAMON SX-3
0,2024-12-27,HA-30/F/20/XC4+XF1/1,6.0,1928.0,6970.0,2100.0,2810.0,0.0,847.0,9.60,13.35
1,2024-12-27,HA-30/F/20/XC4+XF1/1,6.0,1917.0,7070.0,2090.0,2820.0,0.0,845.0,9.60,13.40
2,2024-12-27,HA-30/F/20/XC4+XF1/1,8.0,2560.0,9200.0,2800.0,3770.0,0.0,1127.0,12.80,17.90
3,2024-12-27,HA-30/F/20/XC4+XF1/1,6.0,1923.0,6980.0,2110.0,2830.0,0.0,845.0,9.60,13.50
4,2024-12-27,HA-30/F/20/XC4+XF1/1,6.0,1927.0,6970.0,2100.0,2830.0,0.0,845.0,9.60,13.50
...,...,...,...,...,...,...,...,...,...,...,...
1281,2024-01-05,HA-30/B/20/IIb/1,6.0,1819.0,7160.0,1800.0,3160.0,0.0,876.0,14.50,0.00
1282,2024-01-05,HA-30/B/20/IIb/1,6.0,1821.0,7080.0,1800.0,3170.0,0.0,876.0,14.50,0.00
1283,2024-01-05,HA-30/B/20/IIb/1,6.0,1811.0,7280.0,1810.0,3150.0,0.0,801.0,14.50,0.00
1284,2024-01-05,HA-30/B/20/IIb/1,6.0,1838.0,7284.0,1820.0,3150.0,0.0,798.0,12.04,0.00


In [7]:
# How many non-zero values each column has and what percentage of all rows that represents.
non_zero_counts = (P_B1_24_clean != 0.0).sum()
non_zero_pct = non_zero_counts / len(P_B1_24_clean) * 100

pd.DataFrame({
    "non_zero_count": non_zero_counts,
    "non_zero_pct": non_zero_pct.round(2)
})

,non_zero_count,non_zero_pct
\n \n Fecha,1286,100.00
\n \n Fórmula,1286,100.00
\n \n Volumen,1286,100.00
Cemento\n 4\n CEM III/A 42,1274,99.07
Arido\n 1\n ARENA 04,1274,99.07
Arido\n 2\n ARIDO 8/14,1225,95.26
Arido\n 3\n ARIDO 12/20,1101,85.61
Arido\n 4\n ARENA CORREC,3,0.23
Agua\n 1\n AGUA,1269,98.68
Aditivo\n 1\n AURAMIX 269,1258,97.82


In [8]:
P_B1_24_clean.shape

(1286, 11)

In [9]:
P_B1_24_clean.columns = ["date", "product", "volume", "cement_3a_42", "arid_04", "arid_8_14", "arid_12_20","corrective_02","water","ad_auramix", "ad_dynamon_sx"]

In [10]:
## ## we add a new column where all values will be P_B1 because in the future merging we need to identify from which plant the load is come from.  
# Inserting a new column and value. 
# P_B1_24_clean["plant"] = "P_B1"
# Inserting a column position too. 
P_B1_24_clean.insert(1, "plant", "P_B1")

In [11]:
P_B1_24_clean.columns.tolist()

['date',
 'plant',
 'product',
 'volume',
 'cement_3a_42',
 'arid_04',
 'arid_8_14',
 'arid_12_20',
 'corrective_02',
 'water',
 'ad_auramix',
 'ad_dynamon_sx']

In [12]:
# The final datsets (all plants and both years merged) will have the following columns in this order:
# ['date', 'plant', 'product', 'volume', 'cement_42_5r', 'cement_3a_42', 'cement_4av', 'arid_04', 'arid_8_14', 'arid_12_20', 'water', 'ad_complast_mr', 'ad_dynamon_sx', 'ad_aut_dynamon', 'ad_auramix', 'corrective_02']
final_order = [
    'date',
    'plant',
    'product',
    'volume',
    'cement_3a_42',
    'arid_04',
    'arid_8_14',
    'arid_12_20',
    'water',
    'ad_dynamon_sx',
    'ad_auramix',
    'corrective_02',
]

P_B1_24_clean = P_B1_24_clean[final_order]

In [13]:
# This time, we don't need to check double cement consumption per row because there is just one cement type.

In [14]:
P_B1_24_clean.to_csv(cfg["P_B1_24_ready_csv_path"], index=False, encoding="utf-8-sig")